In [2]:
# !pip install transformers
# !pip install datasets
# !pip install accelerate
# !pip install evaluate

In [3]:
# !pip uninstall -y torchvision
# !pip install torchvision==0.20.1

In [4]:
from transformers import Trainer

In [5]:
import pandas as pd
import numpy as np

import torch

from datasets import Dataset

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
df=pd.read_csv('/content/drive/MyDrive/NLP/AOL/YoutubeCommentsDataSet.csv')

df.dropna(inplace=True)

df.head()

,Comment,Sentiment
0,lets not forget that apple pay in 2014 require...,neutral
1,here in nz 50 of retailers don’t even have con...,negative
2,i will forever acknowledge this channel with t...,positive
3,whenever i go to a place that doesn’t take app...,negative
4,apple pay is so convenient secure and easy to ...,positive


In [8]:
import re

def preprocess_roberta(text):

    text = str(text).lower()

    text = re.sub(
        r"http\S+",
        "",
        text
    )

    text = re.sub(
        r"www\S+",
        "",
        text
    )

    return text.strip()

df["text"] = df["Comment"].apply(
    preprocess_roberta
)

In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["Sentiment"]
)

In [10]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [11]:
train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

In [12]:
tokenizer = RobertaTokenizer.from_pretrained(
    "roberta-base"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [13]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [14]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/14691 [00:00<?, ? examples/s]

Map:   0%|          | 0/3673 [00:00<?, ? examples/s]

In [15]:
train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

In [16]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted"
    )

    acc = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [18]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    fp16=True
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.479375,0.365982,0.855432,0.852079,0.855432,0.851033
2,0.254361,0.360132,0.882657,0.880056,0.882657,0.880580
3,0.170614,0.492099,0.881024,0.880053,0.881024,0.880475


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2757, training_loss=0.2847302491828329, metrics={'train_runtime': 462.3882, 'train_samples_per_second': 95.316, 'train_steps_per_second': 5.963, 'total_flos': 2899049414881536.0, 'train_loss': 0.2847302491828329, 'epoch': 3.0})

In [21]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.170614,0.360132,3,0.882657,0.880056,0.882657,0.880580


{'eval_loss': 0.36013248562812805, 'eval_accuracy': 0.8826572284236319, 'eval_precision': 0.880056343360561, 'eval_recall': 0.8826572284236319, 'eval_f1': 0.8805804982495847}


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import numpy as np

# Prediksi pada test set
predictions = trainer.predict(test_dataset)

# Label aktual
y_true = predictions.label_ids

# Label hasil prediksi
y_pred = np.argmax(predictions.predictions, axis=1)

# Nama kelas
class_names = label_encoder.classes_

# Classification Report
print("CLASSIFICATION REPORT")


print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title("Confusion Matrix - RoBERTa")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [22]:
model.save_pretrained(
    "/content/drive/MyDrive/NLP/AOL/models/roberta_model"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/NLP/AOL/models/roberta_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/NLP/AOL/models/roberta_model/tokenizer_config.json',
 '/content/drive/MyDrive/NLP/AOL/models/roberta_model/tokenizer.json')